In [ ]:
# installing necessary libraries if not done already
!pip install tensorflow

In [ ]:
from google.colab import drive
# importing the necesarry libraries;
# NOTE: tensorflow.keras.preprocessing.image is depricated and no longer
# supported. It is only included here for teaching purposes. Please refer
# to more current up-to-date libraries for image preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import matplotlib.pyplot as plt
import os
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

In [ ]:
# mount your local Google Drive
drive.mount('/content/gdrive')

In [ ]:
# define a path to where on your Google Drive are data, e.g. an example image
path_ = '/content/gdrive/MyDrive/<your local drive>/'
os.chdir(path_)

In [ ]:
# list the content in the directory asigned to "path_"; here we see that the
# directory contains the presentation "Data, hyperparameters and model
# performance", the example image "pexels-rinoadamo-35224529.jpg" and the
# Colab notebook "standardising_images.ipynb"
!ls

# Loading and resizing an image

In [ ]:
# load the example image
img = load_img("pexels-rinoadamo-35224529.jpg")# or tha name of any other image
# print the image type, here 'PIL.JpegImagePlugin.JpegImageFile'
print(type(img))
# print the image format, here "JPEG"
print(img.format)
# print the image mode, here "RGB" for red-green-blue, i.e. a colour image
print(img.mode)
# print the size of the image in pixels, here x or width 9026 by y or height 5156
print(img.size)

In [ ]:
# displaying the example image in matplotlib; each pixel is represented as a
# point in x and y
plt.figure(figsize=(10,10))
plt.imshow(img)

In [ ]:
# resizing the image at the same time as loading it; the original image
# is now resized to 500 by 500 pixels; resizing impacts the resolution of
# the image; however, for vision models standardised image sizes are absolutely
# necessary; resizing is not the same as cropping or zooming into the image
resized = load_img("pexels-rinoadamo-35224529.jpg", target_size=(500, 500))
# print the type of the resized image, here "'PIL.Image.Image'"
print(type(resized))
# print the format of the resized image; as this is now an image object rather
# than a file on disk it no longer has a format, here "None"
print(resized.format)
# print the mode of the resized image, here "RGB" for red-green-blue, i.e. a
# colour image
print(resized.mode)
# print the size of the resized image in pixels, here x or width 500 by y or
# height 500
print(resized.size)
# displaying the resized image in matplotlib; each pixel is represented as a
# point in x and y
plt.figure(figsize=(10,10))
plt.imshow(resized)

# Normalizing pixels

In [ ]:
# turn the pixels in the example image into an array; the array is the first
# step needed to turn the RGB values in the pixels of an image into a
# numerical value that can be read by an algorithm
img_array = img_to_array(img)
# print the shape of the array
print(img_array.shape)
# print the RGB values for all pixels in image 0 in row 0
print(img_array[0][0])
# print the RGB values for all pixels in image 0 in row 3000
print(img_array[0][3000])
# print the RGB values for all pixels in image 0 in row 5000
print(img_array[0][5000])

# turn the array of the example image into a deep learning input that besides
# the RGB values also contains a batch number and a channel count of "3" to
# indicate a colour image
print(80 * "*")
img_array_reshaped = img_array.reshape((1,)+img_array.shape)
# print the shape of the deep-learning prepared array
print(img_array_reshaped.shape)
# print the RGB values for all pixels in image 0 in row 0 in the deep-learning
# prepared array
print(img_array_reshaped[0][0])
# print the RGB values for all pixels in image 0 in row 3000 in the deep-learning
# prepared array
print(img_array_reshaped[0][3000])
# print the RGB values for all pixels in image 0 in row 5000 in the deep-learning
# prepared array
print(img_array_reshaped[0][5000])

In [ ]:
# in order to train a vision model RGB values of an image need to be normalized
# the the full colour range 0 to 255; some RGB values will be close to 0 and
# others close to 255; learning from values spanning different orders of
# magnitude is difficult for a model; so, all RGB values are standardised to a
# value between 0 and 1
# read more about RGB colour model here: https://en.wikipedia.org/wiki/RGB_color_model
normalize_img_array = img_array_reshaped/255.0
# print the normalized RGB values for all pixels in image 0 in row 3000 in the
# deep-learning prepared array
print(normalize_img_array[0][3000])
# print the normalized RGB values for all pixels in image 0 in row 5000 in the
# deep-learning prepared array
print(normalize_img_array[0][5000])

In [ ]:
# repeat the same image-to-array step for the resized image at 500 by 500 pixels
resized_arr = img_to_array(resized)
# expand the resized image array to contain a batch number and colour channels
img_resized_arr_reshaped = resized_arr.reshape((1,)+resized_arr.shape)
# print the shape of the deep-learning prepared array for the resized image
print(img_resized_arr_reshaped.shape)
# print the RGB values for all pixles in row 0 of image 0 in the deep-learning
# prepared array for the resized image
print(img_resized_arr_reshaped[0][0])
# print the RGB values for all pixles in row 450 of image 0 in the deep-learning
# prepared array for the resized image
print(img_resized_arr_reshaped[0][450])



In [ ]:
# in order to train a vision model RGB values of an image need to be normalized
# the the full colour range 0 to 255; some RGB values will be close to 0 and
# others close to 255; learning from values spanning different orders of
# magnitude is difficult for a model; so, all RGB values are standardised to a
# value between 0 and 1
# read more about RGB colour model here: https://en.wikipedia.org/wiki/RGB_color_model
# normalize the RGB values of the resized image
normalize_resized_array = img_resized_arr_reshaped/255.0
# print the normalized RGB values for all pixles in row 0 of image 0 in the
# deep-learning prepared array for the resized image
print(normalize_resized_array[0][0])
# print the normalized RGB values for all pixles in row 450 of image 0 in the
# deep-learning prepared array for the resized image
print(normalize_resized_array[0][450])

# Augmenting images

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# use the ImageDataGenerator to rotate an image by 40deg starting from a
# random angle
datagen1 = ImageDataGenerator(rotation_range=40)
aug_iter1 = datagen1.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly rotated images
for i in range(5):
  batch1 = next(aug_iter1)
  plt.subplot(1,5,i+1)
  plt.imshow(batch1[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# use the ImageDataGenerator to shift an image along its width (x axis) by a
# fraction of 0.2
datagen2 = ImageDataGenerator(width_shift_range=0.2)
aug_iter2 = datagen2.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly shifted images
for i in range(5):
  batch2 = next(aug_iter2)
  plt.subplot(1,5,i+1)
  plt.imshow(batch2[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# use the ImageDataGenerator to shift an image along its height (y axis) by a
# fraction of 0.2
datagen3 = ImageDataGenerator(height_shift_range=0.2)
aug_iter3 = datagen3.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly shifted images
for i in range(5):
  batch3 = next(aug_iter3)
  plt.subplot(1,5,i+1)
  plt.imshow(batch3[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# use the ImageDataGenerator to shear an image by an angle of 40deg
datagen4 = ImageDataGenerator(shear_range=40)
aug_iter4 = datagen4.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly sheared images
for i in range(5):
  batch4 = next(aug_iter4)
  plt.subplot(1,5,i+1)
  plt.imshow(batch4[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# use the ImageDataGenerator to zoom into an image
datagen5 = ImageDataGenerator(zoom_range=[0.3, 1.5])
aug_iter5 = datagen5.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly zoomed images
for i in range(5):
  batch5 = next(aug_iter5)
  plt.subplot(1,5,i+1)
  plt.imshow(batch5[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# NOTE: after applying the vertical flip the cat seems to be horizontally
# flipped; the likely reason is that the original photo was taken so that the
# origin of the image was not on the top left corner
# use the ImageDataGenerator to do a vertical flip of an image
datagen6 = ImageDataGenerator(vertical_flip=True)
aug_iter6 = datagen6.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly flipped images
for i in range(5):
  batch6 = next(aug_iter6)
  plt.subplot(1,5,i+1)
  plt.imshow(batch6[0])
  plt.axis('off')
plt.show()

In [ ]:
# NOTE: each time the code is run the set of images changes; this can be fixed
# by specifying a random seed
# NOTE: after applying the vertical flip the cat seems to be horizontally
# flipped; the likely reason is that the original photo was taken so that the
# origin of the image was not on the top left corner
# use the ImageDataGenerator to do a horizontal flip of an image
datagen7 = ImageDataGenerator(horizontal_flip=True)
aug_iter7 = datagen7.flow(normalize_resized_array, batch_size=1)
plt.figure(figsize=(10,10))
# create 5 randomly flipped images
for i in range(5):
  batch7 = next(aug_iter7)
  plt.subplot(1,5,i+1)
  plt.imshow(batch7[0])
  plt.axis('off')
plt.show()

In [ ]:
datagen_mix = ImageDataGenerator(width_shift_range=[-100, 100],
                    height_shift_range=[-100, 100],
                    rotation_range=120,
                    zoom_range = [0.3, 1.5] ,shear_range=50)
#Creates our batch of one image
pic = datagen_mix.flow(normalize_resized_array, batch_size =1)
plt.figure(figsize=(16, 16))
#Plots our figures
for i in range(1,17):
   plt.subplot(4, 4, i)
   batch = next(pic)
   image_ = batch[0]
   plt.imshow(image_)
plt.show()

# Albuminations as alternative to ImageDataGenerator

In [ ]:
# Tested with albumentations>=2.0

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.7,
            contrast_limit=0.2,
            p=0.5
        ),
        A.ShiftScaleRotate(
            shift_limit=0.3,
            scale_limit=0.05,
            rotate_limit=90,
            border_mode=cv2.BORDER_REFLECT,
            p=0.5
        ),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.1), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.3),
        A.CoarseDropout(
            num_holes_range=(4, 8),
            hole_height_range=(16, 32),
            hole_width_range=(16, 32),
            fill=0,
            p=0.3
        ),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["class_labels"],
    ),
)

# Apply to an image with bounding boxes
image = cv2.imread("pexels-rinoadamo-35224529.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

augmented = transform(
    image=image,
    bboxes=[[50, 60, 200, 300]],
    class_labels=["cat"],
)
augmented_image = augmented["image"]
augmented_bboxes = augmented["bboxes"]

In [ ]:


plt.figure(figsize=(10,10))

# Convert the PyTorch tensor to a NumPy array and transpose the axes
# from (C, H, W) to (H, W, C)
image_np = augmented_image.permute(1, 2, 0).cpu().numpy()

# Define the mean and std used in the albumentations Normalize transform
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Denormalize the image
image_denormalized = image_np * std + mean

# Clip the values to the valid range [0, 1] for displaying float images
image_denormalized = np.clip(image_denormalized, 0, 1)

plt.imshow(image_denormalized)
plt.axis('off')
plt.show()